# Phase B — Segmentation Training on Climbing Holds Dataset

Trains `v9-t-seg` and `v9-s-seg` on the 19-image hand-labelled climbing dataset.

**Prereqs:**
1. Drive folder `My Drive/climbing-holds/` with `data/climbing_holds.zip` and `pretrained/v9-t.ckpt`, `pretrained/v9-s.ckpt` (cached by phase A notebook)
2. `feature/climbing-seg` branch on `https://github.com/ob-choco/YOLO` includes Tasks 10–24 commits

**Output:** seg checkpoints + mask visualizations to `My Drive/climbing-holds/runs/phaseB-<size>-<datetime>/`

**Runtime:** Colab Free T4 → ~3–6 hours per model size.

**Phase A → Phase B weight transfer:** detection backbone/neck/head weights are copied verbatim into the seg model; mask coefficient heads and the prototype Conv are randomly initialized. See `tools/weight_transfer/v9_to_seg.py`.

**Spec / plan refs:**
- `docs/superpowers/specs/2026-05-06-climbing-holds-segmentation-design.md`
- `docs/superpowers/plans/2026-05-06-climbing-holds-segmentation.md` Tasks 25, 26

## 1. Mount Drive and clone the fork

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, subprocess
os.chdir("/content")
if not os.path.exists("YOLO"):
    subprocess.check_call([
        "git", "clone", "-b", "feature/climbing-seg",
        "https://github.com/ob-choco/YOLO.git",
    ])
os.chdir("/content/YOLO")
subprocess.check_call(["git", "pull"])
print(subprocess.check_output(["git", "log", "--oneline", "-5"]).decode())

## 2. Install dependencies

In [ ]:
# Don't reinstall torch / torchvision — Colab ships them with CUDA preinstalled.
!grep -vE "^(torch|torchvision)$" requirements.txt > /tmp/req-no-torch.txt
!pip install -q -r /tmp/req-no-torch.txt
!pip install -q pycocotools

import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("\n❌ NO GPU DETECTED.")
    print("   Colab menu: Runtime → Change runtime type → T4 GPU, then re-run.")
    raise SystemExit("GPU runtime required.")

## 3. Unzip dataset from Drive

In [ ]:
import os, subprocess
data_root = "/content/YOLO/data/climbing_holds"
if not os.path.exists(f"{data_root}/annotations/instances_train.json"):
    !mkdir -p /content/YOLO/data
    !unzip -q -o /content/drive/MyDrive/climbing-holds/data/climbing_holds.zip -d /content/YOLO/
n_train = int(subprocess.check_output(f"ls {data_root}/images/train 2>/dev/null | wc -l", shell=True))
n_val = int(subprocess.check_output(f"ls {data_root}/images/val 2>/dev/null | wc -l", shell=True))
print(f"images/train: {n_train} (slim=15, fat=1061)")
print(f"images/val:   {n_val} (slim=4, fat=1212)")
sample = subprocess.check_output(f"ls {data_root}/images/train | head -1", shell=True).decode().strip()
if sample:
    full = f"{data_root}/images/train/{sample}"
    size = os.path.getsize(full) if os.path.exists(full) else 0
    print(f"First image: {sample} — {size} bytes")
    if size == 0:
        raise SystemExit("Dataset zip stored broken symlinks. Use the slim zip with real files.")
if n_train == 0 or n_val == 0:
    raise SystemExit("Dataset extraction yielded 0 files.")

## 4. Pretrained detection ckpts → seg-init weights

Download (or pull from Drive cache) the v9-t / v9-s detection ckpts, then run `tools.weight_transfer.v9_to_seg` to produce `v9-t-seg-init.pt` / `v9-s-seg-init.pt`. The transfer script copies backbone/neck/head detection weights into `MultiheadSegmentation.detect`; mask coefficient heads and the prototype Conv stay randomly initialized.

In [ ]:
import os, urllib.request, shutil, subprocess
os.makedirs("weights", exist_ok=True)
drive_pretrained = "/content/drive/MyDrive/climbing-holds/pretrained"
os.makedirs(drive_pretrained, exist_ok=True)

# v1.0-alpha release ships detection .pt weights only (no v9-*-seg pretrained).
# We pull the detection ckpts then run our own weight transfer to seed seg models.
for ckpt in ["v9-t.pt", "v9-s.pt"]:
    drive_path = os.path.join(drive_pretrained, ckpt)
    local_path = os.path.join("weights", ckpt)
    if os.path.exists(drive_path):
        shutil.copy(drive_path, local_path)
    else:
        url = f"https://github.com/MultimediaTechLab/YOLO/releases/download/v1.0-alpha/{ckpt}"
        try:
            urllib.request.urlretrieve(url, local_path)
            shutil.copy(local_path, drive_path)
            print(f"Downloaded {ckpt} ({os.path.getsize(local_path)} bytes)")
        except Exception as e:
            print(f"⚠️  {ckpt} unavailable: {e}")

# Run weight transfer for each size (detection → seg-init).
for src, out_name, proto_ch in [
    ("weights/v9-t.pt", "weights/v9-t-seg-init.pt", 128),
    ("weights/v9-s.pt", "weights/v9-s-seg-init.pt", 256),
]:
    if os.path.exists(src):
        subprocess.check_call([
            "python", "-m", "tools.weight_transfer.v9_to_seg",
            "--detection-ckpt", src,
            "--out", out_name,
            "--prototype-channels", str(proto_ch),
        ])
    else:
        print(f"Skipping seg-init for missing {src}; will train from random init.")

## 5a. Train v9-t-seg

100 epochs, batch=8, image_size=640, T4 GPU.

Override flags learned during the laptop smoke test (Task 24):
- `accelerator=gpu device=1`
- `image_size=[640,640]`
- `use_wandb=False`
- `weight=weights/v9-t-seg-init.pt` if available, else `False`

In [ ]:
import datetime, subprocess, os
run_name = f"phaseB-v9-t-seg-{datetime.datetime.now().strftime('%Y%m%d-%H%M')}"
weight_arg = "weights/v9-t-seg-init.pt" if os.path.exists("weights/v9-t-seg-init.pt") else "False"
cmd = (
    "python yolo/lazy.py task=train model=v9-t-seg dataset=climbing_holds "
    "dataset.path=/content/YOLO/data/climbing_holds "
    "task.data.batch_size=8 image_size=[640,640] task.epoch=100 "
    "accelerator=gpu device=1 use_wandb=False "
    f"weight={weight_arg} name={run_name}"
)
print("$", cmd)
result = subprocess.run(cmd + " 2>&1", shell=True, capture_output=True, text=True, bufsize=1)
print(result.stdout or "(no stdout captured)")
if result.returncode != 0:
    raise RuntimeError(f"v9-t-seg training failed with exit code {result.returncode}.")
print("v9-t-seg run:", run_name)

## 5b. Train v9-s-seg

In [ ]:
run_name_s = f"phaseB-v9-s-seg-{datetime.datetime.now().strftime('%Y%m%d-%H%M')}"
weight_arg_s = "weights/v9-s-seg-init.pt" if os.path.exists("weights/v9-s-seg-init.pt") else "False"
cmd_s = (
    "python yolo/lazy.py task=train model=v9-s-seg dataset=climbing_holds "
    "dataset.path=/content/YOLO/data/climbing_holds "
    "task.data.batch_size=8 image_size=[640,640] task.epoch=100 "
    "accelerator=gpu device=1 use_wandb=False "
    f"weight={weight_arg_s} name={run_name_s}"
)
print("$", cmd_s)
result = subprocess.run(cmd_s + " 2>&1", shell=True, capture_output=True, text=True, bufsize=1)
print(result.stdout or "(no stdout captured)")
if result.returncode != 0:
    raise RuntimeError(f"v9-s-seg training failed with exit code {result.returncode}.")
print("v9-s-seg run:", run_name_s)

## 6. Save artifacts to Drive

In [ ]:
import shutil, os
for run in (run_name, run_name_s):
    src = f"/content/YOLO/runs/train/{run}"
    dst = f"/content/drive/MyDrive/climbing-holds/runs/{run}"
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print("Saved:", dst)
    else:
        print("Run dir missing (training failed?):", src)

## 7. Quick visual review

Show the latest val PNG for each seg run. **Note:** the current `ImageLogger.on_validation_batch_end` (Phase A fix) draws bboxes only — for true mask overlays use `yolo.tools.drawer.draw_masks` (Task 23 helper, see below).

In [ ]:
from IPython.display import Image as _Img, display
import glob
for run in (run_name, run_name_s):
    pngs = sorted(glob.glob(f"runs/train/{run}/val_pred_epoch*.png"))
    if pngs:
        print(f"\n=== {run} (latest val bbox PNG) ===")
        display(_Img(pngs[-1]))

## 8. (Optional) Mask overlay visualization

Run a quick inference on the 4 val images using the trained seg model + `draw_masks` to see actual mask overlays. This is post-hoc — not required for the run to succeed.

TODO: Wire this up after the first successful Phase B run; needs the trained model loaded back from `best.ckpt` plus a thin loop that reads val images, runs forward, NMS-filters detections, synthesizes masks via `sigmoid(coef @ proto)`, crops to bbox, and feeds `draw_masks(img, masks, bboxes)`. The smoke test verified all the pieces work; tie them together once we have a meaningfully trained checkpoint.